# Tarea #10 · Separar a los jugadores en dos equipos

**Para qué sirve:** sin saber de qué equipo es cada jugador no podemos saber hacia qué portería ataca nuestro delantero, ni con quién compararlo (compañeros y rivales). En el notebook 09 lo pusimos **a mano** (`ATACA_IZQUIERDA = True`); aquí lo sacamos **automáticamente**.

**Cómo** (con lo visto en clase):
1. **Huellas:** reutilizamos las de la #9 (`embeddings.npy`): ResNet-18 (512 valores) + color del torso (128).
2. **K-Means (no supervisado)** agrupa los jugadores en 2 grupos y nos *propone* 20 recortes de cada uno. Los miramos en un mosaico y corregimos si hay alguno mal → ya tenemos **40 recortes etiquetados** sin etiquetar a mano desde cero.
3. **KNN y regresión logística (supervisado)** entrenados con esos 40 recortes. Los comparamos en *validación* y damos el resultado final en *test* (partes distintas del vídeo).
4. Clasificamos todas las detecciones y **suavizamos** dentro de cada track (mayoría en 1 segundo). Porteros y árbitros los marca el detector.
5. **Sentido de ataque:** cada equipo defiende la portería donde está su portero.

**Antes de empezar:** haber ejecutado `09_reid.ipynb` (usa `embeddings.npy` y `posiciones.csv`). Este notebook no necesita la GPU: tarda segundos.

In [ ]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from sklearn.decomposition import PCA

from src.fase1_equipos import (ClasificadorEquipos, asignar_equipos, clase_de_track,
                               proponer_etiquetas, seleccionar_partes, sentido_de_ataque,
                               equipo_de_un_jugador)
from src.fase1_tracking import color_torso
from src.fase2_homografia import LARGO_M, ANCHO_M

VIDEO = "data/clips/2e57b9_0.mp4"
SALIDA = Path("outputs") / Path(VIDEO).stem
FIG = Path("outputs/figuras"); FIG.mkdir(parents=True, exist_ok=True)

todos = pd.read_csv(SALIDA / "posiciones_todos.csv")
embs = np.load(SALIDA / "embeddings.npy")
assert len(embs) == len(todos), "embeddings.npy no corresponde a posiciones_todos.csv: vuelve a ejecutar 09"
tipo = todos.track_id.map(clase_de_track(todos))
print(f"{len(todos)} detecciones · tracks por tipo:", tipo.groupby(todos.track_id).first().value_counts().to_dict())

In [ ]:
# Utilidad: mosaico de recortes (para mirar con nuestros ojos)
cap = cv2.VideoCapture(VIDEO)
def recorte(r, ancho=60, alto=135, margen=0):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(r.frame)); ok, f = cap.read()
    x1, y1, x2, y2 = int(r.x1), int(r.y1), int(r.x2), int(r.y2)
    m = int(margen * (y2 - y1))
    c = f[max(0, y1 - m):y2 + m, max(0, x1 - m):x2 + m]
    return cv2.cvtColor(cv2.resize(c, (ancho, alto), interpolation=cv2.INTER_NEAREST), cv2.COLOR_BGR2RGB)

def mosaico(filas, titulos, por_fila=20, bordes=None, titulo=""):
    n = len(filas); nf = int(np.ceil(n / por_fila))
    fig, axs = plt.subplots(nf, por_fila, figsize=(por_fila * 0.8, nf * 1.9))
    for ax in np.ravel(axs): ax.axis("off")
    for ax, (_, r), t in zip(np.ravel(axs), filas.iterrows(), titulos):
        ax.imshow(recorte(r)); ax.set_title(t, fontsize=7)
    fig.suptitle(titulo); plt.tight_layout()
    return fig

## 1. K-Means nos propone 40 recortes (20 por grupo)

Solo se usan los **primeros 10 segundos** (fotogramas < 250) para elegir los recortes de entrenamiento; el resto del vídeo queda para validar (10-20 s) y para el test final (20-30 s).

K-Means no sabe qué es "blanco" ni "azul": solo hace dos grupos. Se cogen recortes **típicos** (cerca del centro de su grupo) y de **jugadores distintos**: 20 recortes del mismo jugador no enseñan nada.

👀 **Mira el mosaico.** Si algún recorte está en el grupo equivocado, escribe su número (el `#` del título) en `CORRECCIONES`.

In [ ]:
FIN_TRAIN, FIN_VAL = 250, 500         # fotogramas: train < 250 ≤ validación < 500 ≤ test
prop = proponer_etiquetas(todos, embs, n_por_equipo=20, frame_max=FIN_TRAIN, partes="color")

# Nombre automático de cada grupo según lo clara que es la camiseta (canal V de HSV)
def brillo(r):
    rgb = recorte(r); h = rgb.shape[0]
    return cv2.cvtColor(rgb[int(.15*h):int(.55*h)], cv2.COLOR_RGB2HSV)[..., 2].mean()
brillo_grupo = prop.assign(b=[brillo(r) for _, r in prop.iterrows()]).groupby("grupo").b.mean()
NOMBRES = {int(brillo_grupo.idxmax()): "claro", int(brillo_grupo.idxmin()): "oscuro"}
print("Grupos:", NOMBRES, "· brillo medio del torso:", brillo_grupo.round(0).to_dict())

fig = mosaico(prop, [f"#{i} · t{r.track_id}\n{NOMBRES[r.grupo]}" for i, r in prop.iterrows()],
              titulo="Recortes propuestos por K-Means (arriba un grupo, abajo el otro)")
plt.savefig(FIG / "equipos_propuesta_kmeans.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
CORRECCIONES = {}        # p. ej. {7: "oscuro"} si el recorte #7 es del otro equipo
prop["equipo"] = prop.grupo.map(NOMBRES)
for i, e in CORRECCIONES.items():
    prop.loc[i, "equipo"] = e
print(prop.equipo.value_counts().to_dict(), f"· {prop.track_id.nunique()} jugadores distintos")

## 2. La "verdad" para evaluar

Para medir si el clasificador acierta necesitamos saber el equipo real de cada detección. Lo hacemos **una vez por jugador** (no por fotograma): miramos un recorte de cada track y apuntamos su equipo. El 7 y el 21 cambian de equipo en el fotograma 282-287 (el cambio de número de la #9), así que van por tramos.

👀 Comprueba el mosaico: cada recorte debe coincidir con la lista de abajo.

In [ ]:
jugadores = todos[tipo == "jugador"].groupby("track_id")
filas = []
for tid, g in jugadores:
    for f in ([100, 500] if tid in (7, 21) else [g.frame.iloc[len(g) // 2]]):
        filas.append(g.iloc[(g.frame - f).abs().argmin()])
filas = pd.DataFrame(filas)
mosaico(filas, [f"track {r.track_id}\nf{r.frame}" for _, r in filas.iterrows()], por_fila=12,
        titulo="Un recorte de cada jugador (el 7 y el 21: antes y después del cambio)")
plt.show()

In [ ]:
# Lo que vemos en el mosaico de arriba (blancos = "claro", azules = "oscuro")
CLARO  = [0, 1, 3, 4, 5, 6, 8, 11, 13]
OSCURO = [10, 12, 14, 15, 16, 17, 18, 19, 22]
TRAMOS = {7: ("claro", "oscuro"), 21: ("oscuro", "claro")}   # (antes del 282, después del 287)

def equipo_real(r):
    if r.track_id in TRAMOS:
        if 282 <= r.frame <= 287: return None        # justo durante el cruce: dudoso, no se evalúa
        return TRAMOS[r.track_id][r.frame > 287]
    return "claro" if r.track_id in CLARO else "oscuro" if r.track_id in OSCURO else None

todos["real"] = [equipo_real(r) for r in todos.itertuples()]
ev = (tipo == "jugador") & todos.real.notna()
val = ev & todos.frame.between(FIN_TRAIN, FIN_VAL - 1)
test = ev & (todos.frame >= FIN_VAL)
print(f"Detecciones con equipo conocido: {ev.sum()} · validación {val.sum()} · test {test.sum()}")
print("Etiquetas de entrenamiento distintas de la verdad:", (prop.equipo.to_numpy() != todos.real.to_numpy()[prop.indice]).sum())

## 3. KNN frente a regresión logística

Entrenamos con los 40 recortes y probamos tres huellas: solo **ResNet**, solo **color** y **ambas**. Elegimos el mejor en **validación** y solo entonces miramos el **test** (si eligiéramos mirando el test, el resultado estaría inflado).

In [ ]:
modelos = [("knn", dict(k=1)), ("knn", dict(k=5)), ("logreg", dict(C=1.0)), ("logreg", dict(C=0.1))]
filas = []
for partes in ["resnet", "color", "ambos"]:
    for metodo, kw in modelos:
        clf = ClasificadorEquipos(metodo, partes, **kw).fit(embs[prop.indice], prop.equipo)
        acc = lambda m: (clf.predict(embs[m.to_numpy()]) == todos.real[m]).mean()
        filas.append(dict(huella=partes, modelo=f"{metodo} {list(kw.items())[0][0]}={list(kw.values())[0]}",
                          metodo=metodo, kw=kw, val=acc(val), test=acc(test)))
res = pd.DataFrame(filas)
mejor = res.sort_values("val", ascending=False).iloc[0]
print(f"Mejor en validación: {mejor.huella} + {mejor.modelo} → val {mejor.val:.2%} · TEST {mejor.test:.2%}")
res[["huella", "modelo", "val", "test"]].style.format({"val": "{:.2%}", "test": "{:.2%}"}).highlight_max("val", color="#cde")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
piv = res.pivot(index="modelo", columns="huella", values="val")[["resnet", "color", "ambos"]]
piv.plot.bar(ax=ax, rot=0, color=["#999", "#e08a2c", "#2b6cb0"])
ax.set_ylim(0.9, 1.0); ax.set_ylabel("accuracy en validación"); ax.grid(axis="y", alpha=.3)
ax.set_title("Separar equipos con 40 recortes etiquetados: el color de la camiseta es lo que más ayuda")
plt.savefig(FIG / "equipos_comparacion.png", dpi=150, bbox_inches="tight"); plt.show()
res[["huella", "modelo", "val", "test"]].to_csv(FIG / "equipos_comparacion.csv", index=False)

## 4. Gráfico PCA: ¿se separan los equipos?

PCA reduce las huellas a 2 números para poder dibujarlas. Si los dos equipos forman dos nubes separadas, cualquier clasificador sencillo funciona. A la izquierda solo ResNet; a la derecha ResNet + color.

In [ ]:
rng = np.random.default_rng(0)
m = np.flatnonzero(ev.to_numpy()); m = rng.choice(m, min(3000, len(m)), replace=False)
otros = np.flatnonzero((tipo != "jugador").to_numpy())
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, partes in zip(axs, ["resnet", "ambos"]):
    pca = PCA(2).fit(seleccionar_partes(embs[m], partes))
    p = pca.transform(seleccionar_partes(embs[m], partes)); q = pca.transform(seleccionar_partes(embs[otros], partes))
    for eq, col in [("claro", "#d9d9d9"), ("oscuro", "#1f3b73")]:
        s = todos.real.to_numpy()[m] == eq
        ax.scatter(p[s, 0], p[s, 1], s=6, c=col, edgecolors="k", linewidths=.2, label=f"jugadores {eq}s")
    for t, mk, col in [("portero", "*", "red"), ("arbitro", "x", "#d4b000")]:
        s = tipo.to_numpy()[otros] == t
        ax.scatter(q[s, 0], q[s, 1], s=20, marker=mk, c=col, label=t + "s")
    ax.set_title(f"PCA de las huellas · {'solo ResNet' if partes == 'resnet' else 'ResNet + color'} "
                 f"({pca.explained_variance_ratio_.sum():.0%} de la varianza)")
    ax.legend(markerscale=2, fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "equipos_pca.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Equipo de todas las detecciones (con suavizado)

Usamos el mejor modelo y, dentro de cada track, la **mayoría de 25 fotogramas** (1 s). Corrige fallos sueltos (un recorte tapado o borroso) pero **sigue al cambio de número**: el track 7 tiene que pasar de claro a oscuro en el fotograma ~285.

In [ ]:
clf = ClasificadorEquipos(mejor.metodo, mejor.huella, **mejor.kw).fit(embs[prop.indice], prop.equipo)
eq = asignar_equipos(todos, embs, clf, ventana=25)
for col, nombre in [("equipo_bruto", "sin suavizar"), ("equipo", "con suavizado")]:
    print(f"{nombre:14s}: test {(eq[col][test] == todos.real[test]).mean():.2%} · "
          f"todo el clip {(eq[col][ev] == todos.real[ev]).mean():.2%}")
mal = eq[ev & (eq.equipo != todos.real)]
print("Fallos que quedan por track:", mal.groupby("track_id").size().to_dict())

fig, ax = plt.subplots(figsize=(12, 2.6))
for tid, y0 in [(7, 1), (21, 0)]:
    g = eq[eq.track_id == tid]
    ax.scatter(g.frame, np.full(len(g), y0), c=np.where(g.equipo == "claro", "#d9d9d9", "#1f3b73"),
               s=25, marker="|", linewidths=3)
ax.axvspan(282, 287, color="orange", alpha=.4, label="cambio de número (#9)")
ax.set_yticks([0, 1], ["track 21", "track 7"]); ax.set_xlabel("fotograma"); ax.legend(loc="upper left")
ax.set_title("Equipo asignado a los tracks 7 y 21 (gris = claro, azul = oscuro)")
plt.savefig(FIG / "equipos_cambio_7_21.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Porteros y sentido de ataque

Cada portero se asigna al equipo cuyo centro está más cerca de él. Un equipo **defiende la portería de su portero y ataca la contraria**. Lo comparamos con lo que pusimos a mano en el 09 (`ATACA_IZQUIERDA = True` para los blancos).

In [ ]:
print("Porteros:", eq[eq.es_portero].groupby("track_id").agg(equipo=("equipo", "first"), x_m=("x_m", "median")).round(1).to_dict("index"))
sentido = sentido_de_ataque(eq)
for e, s in sentido.items():
    print(f"  {e:7s} ataca hacia la {s['ataca']} de la cámara (según {s['metodo']})")

TRACK_ID = 7
equipo_jugador = equipo_de_un_jugador(eq, todos[(todos.track_id == TRACK_ID) & (todos.frame < 250)])
ataca_izq_auto = sentido[equipo_jugador]["ataca"] == "izquierda"
print(f"\nNuestro delantero (track {TRACK_ID} al principio) es del equipo '{equipo_jugador}'"
      f" → ATACA_IZQUIERDA automático = {ataca_izq_auto} (a mano pusimos True:"
      f" {'coincide ✅' if ataca_izq_auto else 'NO coincide ❌'})")

In [ ]:
# Un fotograma en el campo 2D, con los equipos y su sentido de ataque
F = 450
pitch = Pitch(pitch_type="custom", pitch_length=LARGO_M, pitch_width=ANCHO_M, line_color="white", pitch_color="#3a7d44")
fig, ax = pitch.draw(figsize=(10, 7))
g = eq[(eq.frame == F) & eq.valido.astype(bool)]
estilos = {"claro": dict(c="white"), "oscuro": dict(c="#1f3b73"), "arbitro": dict(c="#ffd400")}
for e, st in estilos.items():
    s = g[(g.equipo == e) & ~g.es_portero]
    pitch.scatter(s.x_m, s.y_m, ax=ax, s=160, edgecolors="k", zorder=3, label=e, **st)
for _, r in g[g.es_portero].iterrows():
    pitch.scatter(r.x_m, r.y_m, ax=ax, s=260, marker="*", edgecolors="k", zorder=3,
                  c=estilos[r.equipo]["c"], label=f"portero ({r.equipo})")
for e, y in [("claro", 72), ("oscuro", -4)]:
    if e in sentido:
        x0, x1 = (30, 75) if sentido[e]["ataca"] == "derecha" else (75, 30)
        ax.annotate("", xy=(x1, y), xytext=(x0, y), annotation_clip=False,
                    arrowprops=dict(arrowstyle="->", lw=3, color=estilos[e]["c"]))
        ax.text((x0 + x1) / 2, y + (2.5 if y > 34 else -3), f"{e} ataca", ha="center", color="k")
ax.legend(loc="lower left", fontsize=8); ax.set_title(f"Fotograma {F}: equipos en el campo (vista de la cámara)", pad=30)
plt.savefig(FIG / "equipos_campo.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Guardar resultados

In [ ]:
eq[["frame", "track_id", "clase", "equipo", "es_portero", "prob"]].to_csv(SALIDA / "equipos.csv", index=False)
json.dump({"modelo": f"{mejor.huella} + {mejor.modelo}", "accuracy_val": round(float(mejor.val), 4),
           "accuracy_test": round(float(mejor.test), 4), "sentido": sentido,
           "jugador": {"track_id_inicial": TRACK_ID, "equipo": equipo_jugador, "ataca_izquierda": bool(ataca_izq_auto)}},
          open(SALIDA / "equipos.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

pos = pd.read_csv(SALIDA / "posiciones.csv")
pos["equipo"] = equipo_jugador          # la columna que dejamos vacía en el 09
pos.to_csv(SALIDA / "posiciones.csv", index=False)
print("Guardado:", SALIDA / "equipos.csv", "·", SALIDA / "equipos.json", "· equipo en posiciones.csv =", equipo_jugador)

## 8. Vídeo de control: cada jugador con el color de su equipo

In [ ]:
GENERAR_VIDEO = True
if GENERAR_VIDEO:
    colores = {"claro": (255, 255, 255), "oscuro": (140, 60, 20), "arbitro": (0, 215, 255), "otro": (0, 0, 255), "": (0, 0, 255)}
    cap2 = cv2.VideoCapture(VIDEO)
    fps = cap2.get(cv2.CAP_PROP_FPS); w = int(cap2.get(3)); h = int(cap2.get(4))
    out = cv2.VideoWriter(str(SALIDA / "control_equipos.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    por_frame = eq.groupby("frame"); n = 0
    while True:
        ok, f = cap2.read()
        if not ok: break
        if n in por_frame.groups:
            for r in por_frame.get_group(n).itertuples():
                c = colores.get(r.equipo, (0, 0, 255))
                cv2.rectangle(f, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), c, 3 if r.es_portero else 2)
                txt = ("P " if r.es_portero else "") + str(r.track_id)
                cv2.putText(f, txt, (int(r.x1), int(r.y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c, 2)
        out.write(f); n += 1
    cap2.release(); out.release()
    print("Vídeo guardado en", SALIDA / "control_equipos.mp4")